<a href="https://colab.research.google.com/github/Prathama-1/Prosperity-4---Solo42/blob/main/Round5_dataset_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats, signal
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Monte Carlo + statistical helpers
from itertools import combinations

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9,
                     'axes.titlesize': 10, 'axes.labelsize': 9})



In [2]:
# ─── CELL 2: Load Data ───────────────────────────────────────
# Upload prices_round_5_day_{2,3,4}.csv and trades_round_5_day_{2,3,4}.csv to Colab
# then run this cell.

import os

def load_prices():
    frames = []
    for day in [2, 3, 4]:
        fname = f"prices_round_5_day_{day}.csv"
        if not os.path.exists(fname):
            raise FileNotFoundError(f"Missing {fname} — upload it to Colab first")
        df = pd.read_csv(fname, sep=';')
        frames.append(df)
    prices = pd.concat(frames, ignore_index=True)
    prices['global_ts'] = (prices['day'] - 2) * 1_000_000 + prices['timestamp']
    prices['spread']    = prices['ask_price_1'] - prices['bid_price_1']
    prices['mid_price'] = prices['mid_price'].astype(float)
    return prices

def load_trades():
    frames = []
    for day in [2, 3, 4]:
        fname = f"trades_round_5_day_{day}.csv"
        if not os.path.exists(fname):
            raise FileNotFoundError(f"Missing {fname} — upload it to Colab first")
        df = pd.read_csv(fname, sep=';')
        frames.append(df)
    trades = pd.concat(frames, ignore_index=True)
    return trades

prices = load_prices()
trades = load_trades()

ALL_PRODUCTS = sorted(prices['product'].unique())
GROUPS = {
    'GALAXY_SOUNDS':  [p for p in ALL_PRODUCTS if p.startswith('GALAXY')],
    'SLEEP_POD':      [p for p in ALL_PRODUCTS if p.startswith('SLEEP')],
    'MICROCHIP':      [p for p in ALL_PRODUCTS if p.startswith('MICROCHIP')],
    'PEBBLES':        [p for p in ALL_PRODUCTS if p.startswith('PEBBLES')],
    'ROBOT':          [p for p in ALL_PRODUCTS if p.startswith('ROBOT')],
    'UV_VISOR':       [p for p in ALL_PRODUCTS if p.startswith('UV')],
    'TRANSLATOR':     [p for p in ALL_PRODUCTS if p.startswith('TRANS')],
    'PANEL':          [p for p in ALL_PRODUCTS if p.startswith('PANEL')],
    'OXYGEN_SHAKE':   [p for p in ALL_PRODUCTS if p.startswith('OXYGEN')],
    'SNACKPACK':      [p for p in ALL_PRODUCTS if p.startswith('SNACK')],
}

print(f"✅ Loaded {len(prices):,} price rows, {len(trades):,} trade rows")
print(f"   Products: {len(ALL_PRODUCTS)} | Days: {sorted(prices['day'].unique())}")

✅ Loaded 210,150 price rows, 35,385 trade rows
   Products: 50 | Days: [np.int64(2), np.int64(3), np.int64(4)]


In [3]:
# ─── CELL 3: Per-Product Summary Statistics ──────────────────
def compute_stats(prices):
    records = []
    for prod in ALL_PRODUCTS:
        sub = prices[prices['product'] == prod].sort_values('global_ts')
        p   = sub['mid_price'].values
        ret = np.diff(np.log(p))
        spread = sub['spread'].values

        # Basic
        mean_p  = p.mean()
        vol_pct = p.std() / mean_p * 100
        trend   = (p[-1] - p[0]) / p[0] * 100  # total % move days 2-4

        # Returns
        ac1     = pd.Series(ret).autocorr(1)
        sharpe  = ret.mean() / (ret.std() + 1e-10) * np.sqrt(10_000)

        # Spread
        sp_mean = spread.mean()
        sp_pct  = sp_mean / mean_p * 100

        # Regime: trending or mean-reverting (Hurst exponent approx)
        lags  = range(2, 20)
        tau   = [np.std(np.subtract(p[lag:], p[:-lag])) for lag in lags]
        hurst = np.polyfit(np.log(lags), np.log(tau), 1)[0]

        records.append({
            'product': prod,
            'group': next((g for g, pl in GROUPS.items() if prod in pl), 'OTHER'),
            'mean_price': mean_p,
            'vol_pct': vol_pct,
            'total_trend_pct': trend,
            'ac1': ac1,
            'hurst': hurst,
            'sharpe_like': sharpe,
            'spread_mean': sp_mean,
            'spread_pct': sp_pct,
            'price_min': p.min(),
            'price_max': p.max(),
        })

    return pd.DataFrame(records).set_index('product')

stats_df = compute_stats(prices)

# ---- Print ranked summary ----
print("\n{'='*70}")
print("PRODUCT SUMMARY — sorted by |total trend| (best momentum candidates first)")
print('='*70)
display_cols = ['group','mean_price','vol_pct','total_trend_pct','hurst','ac1','spread_pct']
print(stats_df[display_cols].sort_values('total_trend_pct').to_string())


{'='*70}
PRODUCT SUMMARY — sorted by |total trend| (best momentum candidates first)
                                       group    mean_price    vol_pct  total_trend_pct     hurst       ac1  spread_pct
product                                                                                                               
PEBBLES_XS                           PEBBLES   8342.466571  16.609821          -34.935  0.505464  0.011880    0.130364
MICROCHIP_OVAL                     MICROCHIP   8749.483940  13.129343          -32.625  0.481838 -0.016503    0.090265
UV_VISOR_AMBER                      UV_VISOR   8321.422117  14.291905          -31.030  0.485624  0.003828    0.130060
MICROCHIP_RECTANGLE                MICROCHIP   8984.971575  10.349920          -21.255  0.492471 -0.006217    0.089914
ROBOT_IRONING                          ROBOT   9118.201999   9.926617          -19.700  0.480030 -0.028599    0.077696
PEBBLES_S                            PEBBLES   9324.757020   8.107220          -19